In [26]:
import pandas as pd
import numpy as np
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
from skfp.fingerprints import MACCSFingerprint

In [27]:

# ==========================================
# FUNKCJA: Parsowanie hierarchii z pliku .obo
# ==========================================
def parse_obo_hierarchy(file_path):
    """
    Czyta plik .obo i zwraca słownik relacji: { 'Rodzic': ['Dziecko1', 'Dziecko2', ...] }
    """
    parent_child_map = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        current_term = None
        for line in f:
            line = line.strip()
            if line.startswith('[Term]'):
                current_term = None
            elif line.startswith('id:'):
                current_term = line.split('id:')[1].strip()
                if current_term not in parent_child_map:
                    parent_child_map[current_term] = []
            elif line.startswith('is_a:'):
                # Format wpisu to często: "is_a: CHEBI:XXXX ! nazwa"
                parent = line.split('is_a:')[1].split('!')[0].strip()
                if parent not in parent_child_map:
                    parent_child_map[parent] = []
                if current_term:
                    parent_child_map[parent].append(current_term)
    return parent_child_map

In [28]:
# ==========================================
# KROK 1: Wczytanie danych
# ==========================================
print("1. Wczytywanie danych...")
train_df = pd.read_parquet('data/chebi_dataset_train.parquet')
test_df = pd.read_parquet('data/chebi_dataset_test_empty.parquet')

# Wyodrębnienie kolumn: SMILES i etykiet klas (zakładamy, że reszta to klasy CHEBI)
smiles_train = train_df['SMILES'].tolist()
y_train = train_df.drop(columns=['SMILES', 'mol_id']).values
class_names = train_df.drop(columns=['SMILES', 'mol_id']).columns.tolist()

smiles_test = test_df['SMILES'].tolist()

1. Wczytywanie danych...


In [29]:

# ==========================================
# KROK 2: Ekstrakcja cech (scikit-fingerprints)
# ==========================================
print("2. Generowanie wektorów MACCS Fingerprint...")
maccs = MACCSFingerprint()

# Transformacja SMILES na wektory 0/1
X_train = maccs.transform(smiles_train)
X_test = maccs.transform(smiles_test)

2. Generowanie wektorów MACCS Fingerprint...


[16:38:49] WARNING: not removing hydrogen atom without neighbors
[16:38:49] WARNING: not removing hydrogen atom without neighbors
[16:38:49] WARNING: not removing hydrogen atom without neighbors
[16:38:49] WARNING: not removing hydrogen atom without neighbors
[16:38:49] WARNING: not removing hydrogen atom without neighbors
[16:38:50] WARNING: not removing hydrogen atom without neighbors
[16:38:50] WARNING: not removing hydrogen atom without neighbors
[16:38:50] WARNING: not removing hydrogen atom without neighbors
[16:38:50] WARNING: not removing hydrogen atom without neighbors
[16:38:50] Unusual charge on atom 0 number of radical electrons set to zero
[16:38:50] WARNING: not removing hydrogen atom without neighbors
[16:38:50] WARNING: not removing hydrogen atom without neighbors
[16:38:50] WARNING: not removing hydrogen atom without neighbors
[16:38:50] WARNING: not removing hydrogen atom without neighbors
[16:38:50] WARNING: not removing hydrogen atom without neighbors
[16:38:50] WAR

In [30]:
# ==========================================
# KROK 3: Trening modelu (Multitask)
# ==========================================
print("3. Trenowanie modelu bazowego...")

# Znajdujemy kolumny, które mają więcej niż 1 unikalną wartość (nie są stałe)
valid_cols_mask = np.array([len(np.unique(y_train[:, i])) > 1 for i in range(y_train.shape[1])])

# Zapisujemy klasy, które są stałe, żeby dodać je z powrotem na końcu
constant_classes = {class_names[i]: y_train[0, i] for i in range(len(class_names)) if not valid_cols_mask[i]}
print(f"   -> Pomijanie {len(constant_classes)} stałych klas (np. rdzeń ontologii)...")

# Filtrujemy dane treningowe tylko do zmiennych klas
y_train_filtered = y_train[:, valid_cols_mask]
valid_class_names = [class_names[i] for i in range(len(class_names)) if valid_cols_mask[i]]

# Trenujemy model na przefiltrowanych danych
base_model = LogisticRegression(class_weight='balanced', max_iter=1000)
model = MultiOutputClassifier(base_model, n_jobs=-1)
model.fit(X_train, y_train_filtered)

print("4. Generowanie predykcji...")
proba_preds_list = model.predict_proba(X_test)

proba_preds_filtered = np.column_stack([
    p[:, 1] if p.shape[1] == 2 else p[:, 0]
    for p in proba_preds_list
])

# Odtwarzamy tabelę z predykcjami dla działających klas
preds_df = pd.DataFrame(proba_preds_filtered, columns=valid_class_names)

# Przywracamy stałe klasy na sztywno (z wartością 1.0 lub 0.0)
for cls_name, const_val in constant_classes.items():
    preds_df[cls_name] = float(const_val)

3. Trenowanie modelu bazowego...
   -> Pomijanie 1 stałych klas (np. rdzeń ontologii)...
4. Generowanie predykcji...


In [31]:

# ==========================================
# KROK 4: Post-processing (Logika DAG)
# ==========================================
print("5. Wgrywanie hierarchii i naprawa niekonsekwencji...")
# Pamiętaj, aby plik chebi_classes.obo był w odpowiednim folderze!
parent_child_map = parse_obo_hierarchy('data/chebi_classes.obo')

# Eliminacja niekonsekwencji: jeśli prawdopodobieństwo dziecka > rodzica, zrównaj w dół
corrections_made = 0
for parent, children in parent_child_map.items():
    if parent in class_names:
        for child in children:
            if child in class_names:
                # Znajdź miejsca, gdzie model dał dziecku wyższy wynik niż rodzicowi [cite: 26]
                mask = preds_df[child] > preds_df[parent]
                if mask.any():
                    preds_df.loc[mask, child] = preds_df.loc[mask, parent]
                    corrections_made += mask.sum()

print(f"   -> Wprowadzono {corrections_made} poprawek logicznych.")

5. Wgrywanie hierarchii i naprawa niekonsekwencji...
   -> Wprowadzono 2064332 poprawek logicznych.


In [33]:

# ==========================================
# KROK 5: Zapis do pliku
# ==========================================
print("6. Binarne formatowanie i zapis...")
# Próg 0.5 zamienia prawdopodobieństwa na 0 i 1
final_preds_binary = (preds_df >= 0.5).astype(int)

submission = pd.DataFrame({'mol_id': test_df['mol_id']})
submission = pd.concat([submission, final_preds_binary], axis=1)

submission.to_parquet('my_submission.parquet', index=False)
print("Gotowe! Plik zapisany jako 'my_submission.parquet'.")




6. Binarne formatowanie i zapis...
Gotowe! Plik zapisany jako 'my_submission.parquet'.
